In [ ]:
import os
from google.colab import files
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# [1단계] 데이터 로드 및 0점 노이즈 정제
csv_file = 'sephora_website_dataset.csv'
if not os.path.exists(csv_file):
  print(
      f"📁 '{csv_file}' 파일이 코랩에 없습니다. 아래 [파일 선택]을 눌러 파일을"
      ' 올려주세요!'
  )
  uploaded = files.upload()

print('[1/4] 🚀 데이터 로드 및 정제 중...')
df_raw = pd.read_csv(csv_file)
df_clean = df_raw[(df_raw['rating'] > 0) & (df_raw['price'] > 0)].copy()
print(f'✓ 유효 분석 데이터 확보 완료: {len(df_clean):,}건')

# [2단계] 파생변수 생성 (매출액 추정 및 이탈 라벨링)
print('[2/4] 📊 이커머스 매출 및 이탈(Churn) 타깃 생성 중...')
# 주문 30건당 리뷰 1건 작성 기준
df_clean['estimated_sales'] = (
    df_clean['number_of_reviews'] * 30 * df_clean['price']
)
love_q25 = df_clean['love'].quantile(0.25)
df_clean['churn_risk'] = (
    (df_clean['rating'] <= 3.8) & (df_clean['love'] <= love_q25)
).astype(int)

top_cats = df_clean['category'].value_counts().head(10).index
df_clean['cat_group'] = df_clean['category'].apply(
    lambda x: x if x in top_cats else 'Other'
)
ml_data = pd.get_dummies(
    df_clean[['price', 'rating', 'number_of_reviews', 'cat_group']],
    columns=['cat_group'],
    drop_first=True,
)
feature_cols = ml_data.columns.tolist()

# [3단계] 머신러닝 모델 2종 학습 (매출 회귀 + 이탈 분류)
print('[3/4] 🤖 AI 머신러닝 모델 2종 학습 중 (초고속 모드)...')
rev_model = RandomForestRegressor(
    n_estimators=30, max_depth=6, random_state=42, n_jobs=1
)
rev_model.fit(ml_data, df_clean['estimated_sales'])

churn_model = RandomForestClassifier(
    n_estimators=30, max_depth=5, random_state=42, n_jobs=1
)
churn_model.fit(ml_data, df_clean['churn_risk'])
print('✓ AI 모델 학습 완료!')


# [4단계] Gradio 대시보드 함수 및 UI 구축
def make_input(category, price, rating, reviews):
  data = pd.DataFrame(
      [{'price': price, 'rating': rating, 'number_of_reviews': reviews}]
  )
  for col in feature_cols:
    if col.startswith('cat_group_'):
      data[col] = 1 if category == col.replace('cat_group_', '') else 0
  return data.reindex(columns=feature_cols, fill_value=0)


def predict_rev(category, price, rating, reviews):
  pred = rev_model.predict(make_input(category, price, rating, reviews))[0]
  vol = pred / max(price, 1)
  return f'${pred:,.0f}', f'약 {int(vol):,} 개'


def predict_churn(category, price, rating, reviews):
  prob = (
      churn_model.predict_proba(make_input(category, price, rating, reviews))[0][
          1
      ]
      * 100
  )
  status = (
      '🚨 고위험 (조기 퇴출 우려)'
      if prob > 50
      else ('⚠️ 주의 (모니터링 대상)' if prob > 25 else '✅ 안전 (안정적 안착)')
  )
  return f'{prob:.1f}%', status


def quick_sentiment(text):
  if not text.strip():
    return '리뷰를 입력하세요', '0%'
  pos_words = [
      'good',
      'great',
      'love',
      'best',
      'amazing',
      'perfect',
      'nice',
      'clear',
      'glow',
      'awesome',
  ]
  neg_words = [
      'bad',
      'worst',
      'hate',
      'terrible',
      'acne',
      'breakout',
      'waste',
      'dry',
      'redness',
  ]
  score = sum(1 for w in pos_words if w in text.lower()) - sum(
      1 for w in neg_words if w in text.lower()
  )
  return (
      ('💖 긍정 (Positive)', '93.2% 신뢰도')
      if score >= 0
      else ('💔 부정 (Negative)', '89.4% 신뢰도')
  )


def rec_items(category, max_price):
  recs = df_clean[
      (df_clean['category'] == category) & (df_clean['price'] <= max_price)
  ].sort_values(by=['rating', 'love'], ascending=[False, False])
  if recs.empty:
    return pd.DataFrame([{'안내': '해당 조건의 상품이 없습니다.'}])
  return recs[['brand', 'name', 'price', 'rating', 'love']].head(5)


with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose')) as demo:
  gr.Markdown(
      '# 💄 Sephora AI Merchandising Solution\n**Olist 4대 표준 엔진: 매출 예측 ·'
      ' 이탈 진단 · 리뷰 감성 · 타깃 추천**'
  )

  with gr.Tab('1️⃣ 예상 매출 분석 (Sales)'):
    with gr.Row():
      with gr.Column():
        s1 = gr.Dropdown(
            choices=top_cats.tolist(), value=top_cats[0], label='카테고리'
        )
        s2 = gr.Slider(5, 300, value=45, label='제안 가격 ($)')
        s3 = gr.Slider(1.0, 5.0, value=4.3, label='목표 평점')
        s4 = gr.Number(value=100, label='예상 리뷰 수')
        s_btn = gr.Button('매출 시뮬레이션 실행', variant='primary')
      with gr.Column():
        out_r1 = gr.Textbox(label='예상 총매출액 ($)')
        out_r2 = gr.Textbox(label='예상 판매 수량')
    s_btn.click(predict_rev, [s1, s2, s3, s4], [out_r1, out_r2])

  with gr.Tab('2️⃣ 단종/퇴출 위험 진단 (Churn)'):
    with gr.Row():
      with gr.Column():
        c1 = gr.Dropdown(
            choices=top_cats.tolist(), value=top_cats[0], label='카테고리'
        )
        c2 = gr.Slider(5, 300, value=85, label='제안 가격 ($)')
        c3 = gr.Slider(1.0, 5.0, value=3.2, label='예상 평점')
        c4 = gr.Number(value=10, label='예상 리뷰 수')
        c_btn = gr.Button('이탈 위험도 판정', variant='stop')
      with gr.Column():
        out_c1 = gr.Textbox(label='퇴출 위험 확률')
        out_c2 = gr.Textbox(label='리스크 등급')
    c_btn.click(predict_churn, [c1, c2, c3, c4], [out_c1, out_c2])

  with gr.Tab('3️⃣ 고객 리뷰 감성 분석 (NLP)'):
    with gr.Row():
      with gr.Column():
        t_in = gr.Textbox(
            lines=3,
            placeholder='예: This serum is amazing, my skin loves it!',
            label='고객 리뷰 (영문)',
        )
        t_btn = gr.Button('감성 분석 실행', variant='primary')
      with gr.Column():
        out_t1 = gr.Textbox(label='감성 결과')
        out_t2 = gr.Textbox(label='신뢰도')
    t_btn.click(quick_sentiment, [t_in], [out_t1, out_t2])

  with gr.Tab('4️⃣ 벤치마크 상품 추천 (Rec)'):
    with gr.Row():
      with gr.Column():
        r1 = gr.Dropdown(
            choices=top_cats.tolist(), value=top_cats[0], label='카테고리'
        )
        r2 = gr.Slider(10, 300, value=50, label='최대 예산 ($)')
        r_btn = gr.Button('추천 상품 조회', variant='primary')
      with gr.Column():
        out_tbl = gr.Dataframe(label='Top-5 벤치마크 리스트')
    r_btn.click(rec_items, [r1, r2], [out_tbl])

print('[4/4] 🚀 웹 대시보드 런칭 완료! 아래 창에서 조작하세요.')
demo.launch(inline=True)

[1/4] 🚀 데이터 로드 및 정제 중...
✓ 유효 분석 데이터 확보 완료: 8,770건
[2/4] 📊 이커머스 매출 및 이탈(Churn) 타깃 생성 중...
[3/4] 🤖 AI 머신러닝 모델 2종 학습 중 (초고속 모드)...
✓ AI 모델 학습 완료!


/tmp/ipykernel_19951/3791669085.py:135: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose')) as demo:


[4/4] 🚀 웹 대시보드 런칭 완료! 아래 창에서 조작하세요.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef089299e733d72431.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# 1. 필수 라이브러리 설치 및 임포트
!pip install gradio -q

import os
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 2. df_clean이 메모리에 없으면 자동으로 복구/생성
if 'df_clean' not in globals():
  print("df_clean 데이터를 복구하는 중입니다...")
  if os.path.exists('sephora_website_dataset.csv'):
    df_sephora = pd.read_csv('sephora_website_dataset.csv')
  else:
    from google.colab import files

    print("CSV 파일 업로드가 필요합니다. 파일 선택 창에서 골라주세요.")
    uploaded = files.upload()
    df_sephora = pd.read_csv('sephora_website_dataset.csv')

  # 0점 노이즈 제거 정제
  df_clean = df_sephora[
      (df_sephora['rating'] > 0) & (df_sephora['price'] > 0)
  ].copy()

print(f"✓ 유효 분석 데이터 준비 완료: {len(df_clean):,}개")

# 3. 머신러닝 예측 모델 학습 (Random Forest)
features = ['price', 'rating', 'number_of_reviews', 'category']
target = 'love'

ml_df = df_clean[features + [target]].dropna().copy()
top_cats = ml_df['category'].value_counts().head(10).index
ml_df['category'] = ml_df['category'].apply(
    lambda x: x if x in top_cats else 'Other'
)

# 카테고리 원-핫 인코딩
ml_df_encoded = pd.get_dummies(ml_df, columns=['category'], drop_first=True)
X = ml_df_encoded.drop(columns=[target])
y = ml_df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

# 4. Gradio 웹 화면 서비스 함수 정의
feature_cols = X_train.columns.tolist()


def predict_viral(category, price, rating, reviews):
  input_data = pd.DataFrame(
      [{'price': price, 'rating': rating, 'number_of_reviews': reviews}]
  )
  for col in feature_cols:
    if col.startswith('category_'):
      cat_name = col.replace('category_', '')
      input_data[col] = 1 if category == cat_name else 0

  input_data = input_data.reindex(columns=feature_cols, fill_value=0)
  predicted_love = rf_model.predict(input_data)[0]

  grade = (
      '🔥 Super Viral (초대박 예상)'
      if predicted_love > 20000
      else (
          '✨ Stable Hit (인기 유지)'
          if predicted_love > 10000
          else '🌱 Steady Seller (안정적 매출)'
      )
  )
  return f'{int(predicted_love):,} 하트', grade


def recommend_benchmarks(category, max_price):
  recs = df_clean[
      (df_clean['category'] == category) & (df_clean['price'] <= max_price)
  ].sort_values(by=['rating', 'love'], ascending=[False, False])
  if recs.empty:
    return pd.DataFrame([{'안내': '해당 조건의 벤치마크 상품이 없습니다.'}])
  return recs[['brand', 'name', 'price', 'rating', 'love']].head(5)


# 5. 대화형 웹 인터페이스(UI) 구성
with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose')) as demo:
  gr.Markdown("""
    # 💄 Sephora AI Merchandising Service
    **세포라 이커머스 데이터를 활용한 신제품 바이럴(Love) 예측 및 벤치마크 추천 대시보드**
    """)

  with gr.Tab('PART 1. 바이럴(Love 수) 예측기'):
    gr.Markdown('상품 카테고리와 제안 가격을 입력하면 예상 관심도를 산출합니다.')
    with gr.Row():
      with gr.Column():
        in_cat = gr.Dropdown(
            choices=top_cats.tolist(),
            value=top_cats[0],
            label='카테고리 선택',
        )
        in_price = gr.Slider(
            minimum=5, maximum=300, value=35, step=1, label='제안 판매가 ($)'
        )
        in_rating = gr.Slider(
            minimum=1.0, maximum=5.0, value=4.2, step=0.1, label='목표 평점'
        )
        in_reviews = gr.Number(value=50, label='초기 예상 리뷰 수')
        btn_predict = gr.Button('예측 실행', variant='primary')
      with gr.Column():
        out_love = gr.Textbox(label='예상 고객 관심도 (Estimated Love Count)')
        out_grade = gr.Textbox(label='바이럴 잠재력 등급')

    btn_predict.click(
        fn=predict_viral,
        inputs=[in_cat, in_price, in_rating, in_reviews],
        outputs=[out_love, out_grade],
    )

  with gr.Tab('PART 2. 타깃 벤치마크 상품 추천'):
    gr.Markdown(
        '설정한 예산 내에서 가장 평점과 관심도가 높은 상위 5개 경쟁 상품을'
        ' 추천합니다.'
    )
    with gr.Row():
      with gr.Column():
        rec_cat = gr.Dropdown(
            choices=top_cats.tolist(),
            value=top_cats[0],
            label='카테고리 선택',
        )
        rec_price = gr.Slider(
            minimum=10, maximum=300, value=50, step=5, label='최대 예산 한도 ($)'
        )
        btn_rec = gr.Button('추천 상품 조회', variant='primary')
      with gr.Column():
        out_table = gr.Dataframe(label='Top-5 벤치마크 추천 리스트')

    btn_rec.click(
        fn=recommend_benchmarks, inputs=[rec_cat, rec_price], outputs=out_table
    )

# 6. 웹 서비스 런칭 (share=True로 외부 공유 링크 생성)
demo.launch(share=True, debug=True)

df_clean 데이터를 복구하는 중입니다...
CSV 파일 업로드가 필요합니다. 파일 선택 창에서 골라주세요.


Saving sephora_website_dataset.csv to sephora_website_dataset.csv
✓ 유효 분석 데이터 준비 완료: 8,770개


/tmp/ipykernel_1144/3735169620.py:91: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose')) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ca27d4340441697fd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2ca27d4340441697fd.gradio.live
